# Objective 1: Sentinel-2 Super-Resolution (4x) -- SwinIR GPU Fine-Tuning
**Project:** AI-Based Forest Cover Segmentation and Temporal Deforestation Detection Using U-Net & SwinIR  
**Target:** 4x Super-Resolution of Sentinel-2 L2A Imagery (10 m GSD -> 2.5 m GSD) using the SEN2NAIP Cross-Sensor Dataset.

---

### Overview & Motivation
The baseline system incorporates an off-the-shelf **SwinIR-M classical super-resolution** model pretrained on DF2K (natural photographs). While this provides 4x spatial upscaling, satellite imagery possesses unique characteristics:
- **Atmospheric & sensor physics**: Sentinel-2 L2A surface reflectance is scaled in [0, 10000] (mean terrestrial reflectance ~0.07-0.10).
- **Target sensor differences**: High-resolution reference imagery (NAIP aerial photography at 2.5 m) has standard RGB 8-bit dynamic range [0, 255] (mean intensity ~0.40-0.50).
- **Generic SR degradation**: Because generic SwinIR expects natural photos, it outputs dark reflectance predictions when evaluated directly against NAIP, leading to baseline PSNR ~9.8 dB.

**Solution:** Fine-tune SwinIR on paired **SEN2NAIP** Sentinel-2 (10 m) and NAIP (2.5 m) cross-sensor imagery using Google Colab GPU acceleration.

---

### What this Notebook Does:
1. **Checks GPU Environment**: Verifies CUDA availability, device specs (e.g., T4/V100/A100), and VRAM.
2. **Clones Project Code**: Clones the repository (`main` branch) and configures python paths.
3. **Installs Dependencies**: Installs `tifffile`, `huggingface_hub`, `scikit-image`, `timm`, and `matplotlib`.
4. **Fixes GeoTIFF Warning**: Resolves `<tifffile.TiffPage ... parsing GDAL_NODATA tag raised ValueError>` by suppressing non-fatal tag parsing notices and establishing a reliable reading pipeline.
5. **Acquires Dataset**: Streams a 100-300 pair subset (default: 150 pairs) directly from Hugging Face via HTTP range requests without downloading the entire 165+ GB dataset (or accepts ZIP / Google Drive).
6. **Partitions Dataset**: Splits into deterministic Train (80%), Validation (10%), and Test (10%) sets.
7. **Verifies Shapes & Window Divisibility**: Confirms 120x120 LR -> 480x480 HR (both divisible by window size 8).
8. **Loads Base Pretrained Model**: Loads `models/swinir_x4.pth` (with fallback auto-download).
9. **Computes Baselines**: Evaluates Bicubic upsampling and Generic SwinIR on the validation set.
10. **Runs GPU Fine-Tuning**: Trains with L1 reconstruction loss and Adam optimizer for 5 epochs.
11. **Saves Fine-Tuned Checkpoint**: Exports to `models/swinir_sentinel_x4.pth` with full metadata (leaving production weights untouched).
12. **Generates 5-Panel Visualizations**: LR, Bicubic, Generic SwinIR, Fine-Tuned SwinIR, and NAIP HR Ground Truth.
13. **Outputs Comparison Table**: Clear quantitative benchmark table (Delta PSNR, Delta SSIM, Delta L1).

> **Before Starting:** Ensure GPU is enabled:  
> **Runtime** -> **Change runtime type** -> Select **T4 GPU** (or A100/L4) -> **Save**.


## 1. Check GPU & Hardware Acceleration


In [ ]:
import os
import sys
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU Name:           {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version:       {torch.version.cuda}")
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024**3)
    print(f"Total VRAM:         {vram_gb:.2f} GB")
    print(f"Compute Capability: {props.major}.{props.minor}")
    print("[SUCCESS] GPU hardware acceleration is active and ready.")
else:
    print("[WARNING] No GPU detected! Running on CPU will be slow (~18s/step).")
    print("Please enable GPU: Runtime > Change runtime type > T4 GPU > Save.")

# Set deterministic seed
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


## 2. Clone Project Repository & Configure Environment


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/vikaskmalipatil/Deforestation-Detection-from-Satellite-Images.git"
BRANCH   = "main"
PROJ_DIR = Path("/content/deforestation")

if not PROJ_DIR.exists():
    print(f"Cloning {BRANCH} branch from {REPO_URL}...")
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJ_DIR)], check=True)
    print(f"[SUCCESS] Cloned project to {PROJ_DIR}")
else:
    print(f"Project directory already exists at {PROJ_DIR}.")

# Navigate to project directory
os.chdir(PROJ_DIR)
print(f"Current Working Directory: {os.getcwd()}")

# Configure sys.path
for p in [str(PROJ_DIR), str(PROJ_DIR / "src"), str(PROJ_DIR / "src" / "swinir")]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("System paths configured successfully.")


## 3. Install Dependencies


In [ ]:
import subprocess
import sys

pkgs = [
    "tifffile",
    "huggingface_hub",
    "scikit-image",
    "timm",
    "matplotlib",
]
print("Installing required packages: " + ", ".join(pkgs))
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs, check=True)

import tifffile
import huggingface_hub
import skimage
import timm
import matplotlib
import numpy as np

print(f"torch:           {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"tifffile:        {tifffile.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")
print(f"scikit-image:    {skimage.__version__}")
print(f"timm:            {timm.__version__}")
print(f"numpy:           {np.__version__}")
print(f"matplotlib:      {matplotlib.__version__}")
print("[SUCCESS] All required dependencies installed and verified.")


## 4. Fix GeoTIFF Warning & Configure Dataset Pipeline

### Technical Analysis of the TIFF-Reading Warning:
When reading GeoTIFF files from SEN2NAIP with `tifffile`, the following warning was observed locally:
`<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '-2.147483648E9'")`

**Cause:**
- Standard GDAL encodes the NoData value in TIFF tag 42113 (`GDAL_NODATA`) as an ASCII string in scientific notation (`"-2.147483648E9"`).
- `tifffile`'s tag parser attempts to cast this string directly using Python's `int()`, which fails on floating-point scientific notation.
- `tifffile` catches this exception internally and emits a warning through Python's standard `logging` framework under the logger name `"tifffile"`.
- The pixel array itself is completely intact and accurately parsed.

**Fix:**
1. Configure `logging.getLogger("tifffile").setLevel(logging.ERROR)` to suppress this cosmetic tag parsing message while maintaining genuine I/O error reporting.
2. Provide a clean helper `read_geotiff()` that reads arrays safely without console clutter.
3. Ensure `src/sen2naip_dataset.py` is present with synchronized cropping, divisibility handling, and normalization.


In [ ]:
import logging
import tifffile
import numpy as np
from pathlib import Path

# Fix: Suppress non-fatal GDAL_NODATA tag parsing warnings
logging.getLogger("tifffile").setLevel(logging.ERROR)

def read_geotiff(path):
    """
    Safely reads a GeoTIFF array with tifffile, cleanly handling GDAL metadata.
    Returns float32 numpy array.
    """
    return tifffile.imread(str(path)).astype(np.float32)

# Verify / Write src/sen2naip_dataset.py into the workspace
dataset_file = Path("src/sen2naip_dataset.py")
dataset_file.parent.mkdir(parents=True, exist_ok=True)
dataset_code = "\"\"\"\nsrc/sen2naip_dataset.py\n=======================\nPyTorch Dataset and DataLoader for Sentinel-2 / NAIP super-resolution fine-tuning.\n\nDataset Properties (SEN2NAIP Cross-Sensor):\n  - LR: Sentinel-2 L2A Surface Reflectance (10 m GSD, 121 x 121 x 4)\n        Bands: [B04 (Red), B03 (Green), B02 (Blue), B08 (NIR)]\n        Native dtype: int32 (scaled x10000)\n  - HR: NAIP Aerial Imagery (2.5 m GSD, 484 x 484 x 4)\n        Bands: [Red, Green, Blue, NIR]\n        Native dtype: uint8 [0, 255]\n  - Spatial Scale Factor: exactly 4.0x (121 x 4 = 484)\n\nBand Selection Policy:\n  - Default: `rgb_only=True` extracts the first 3 channels (Red, Green, Blue).\n    Rationale: Our existing SwinIR architecture (SwinIR-M classical SR) is parameterized\n    for 3-channel RGB input/output, and the GIS web dashboard displays true-color RGB.\n  - Setting `rgb_only=False` returns all 4 channels [R, G, B, NIR].\n    The NIR band is preserved on disk in the GeoTIFF files and is never silently deleted.\n\nSpatial Alignment & SwinIR Window Divisibility:\n  - SwinIR uses window-based self-attention with window_size=8.\n  - The native LR size 121x121 is not divisible by 8 (121 % 8 = 1).\n  - By default, this loader applies a crop of (120, 120) for LR and (480, 480) for HR,\n    which satisfies divisibility for both LR (120 % 8 = 0) and HR (480 % 32 = 0)\n    while preserving the exact 4x spatial scale.\n\"\"\"\n\nimport json\nimport random\nfrom pathlib import Path\nfrom typing import Tuple, Optional, Dict, List\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\n\ntry:\n    import logging\n    import tifffile\n    logging.getLogger(\"tifffile\").setLevel(logging.ERROR)\nexcept ImportError:\n    raise ImportError(\"tifffile is required for SEN2NAIP GeoTIFF loading. Install via: pip install tifffile\")\n\nDEFAULT_DATASET_DIR = Path(__file__).resolve().parent.parent / \"data\" / \"SEN2NAIP\"\n\n\nclass SEN2NAIPDataset(Dataset):\n    \"\"\"\n    PyTorch Dataset for paired Sentinel-2 (LR) and NAIP (HR) super-resolution.\n\n    Args:\n        root_dir:     Path to data/SEN2NAIP directory.\n        split:        One of 'train', 'val', 'test', or 'all'.\n        splits_file:  Optional path to splits.json. Defaults to root_dir / 'splits.json'.\n        rgb_only:     If True, returns 3-channel RGB [R, G, B].\n                      If False, returns 4-channel [R, G, B, NIR].\n        lr_crop_size: LR crop dimensions (H, W). Default (120, 120) to ensure divisibility by 8.\n                      Set to None to keep native 121x121.\n        scale:        Super-resolution scale factor (default: 4).\n        random_crop:  If True, selects random crop location (recommended for training).\n                      If False, selects center crop (deterministic for val/test).\n        augment:      If True, applies synchronized spatial augmentations (flips, rotations).\n    \"\"\"\n\n    def __init__(\n        self,\n        root_dir: Path = DEFAULT_DATASET_DIR,\n        split: str = \"train\",\n        splits_file: Optional[Path] = None,\n        rgb_only: bool = True,\n        lr_crop_size: Optional[Tuple[int, int]] = (120, 120),\n        scale: int = 4,\n        random_crop: Optional[bool] = None,\n        augment: Optional[bool] = None,\n    ):\n        self.root_dir = Path(root_dir)\n        self.cross_dir = self.root_dir / \"cross-sensor\"\n        self.split = split\n        self.rgb_only = rgb_only\n        self.lr_crop_size = lr_crop_size\n        self.scale = scale\n\n        # Defaults for cropping and augmentation based on split\n        self.random_crop = (split == \"train\") if random_crop is None else random_crop\n        self.augment = (split == \"train\") if augment is None else augment\n\n        if not self.cross_dir.exists():\n            raise FileNotFoundError(f\"SEN2NAIP cross-sensor directory not found at {self.cross_dir}\")\n\n        # Resolve splits\n        splits_path = Path(splits_file) if splits_file else self.root_dir / \"splits.json\"\n        if splits_path.exists():\n            with open(splits_path, \"r\", encoding=\"utf-8\") as f:\n                splits_data = json.load(f)\n            if split == \"all\":\n                self.roi_ids = sorted(\n                    splits_data.get(\"train\", []) + splits_data.get(\"val\", []) + splits_data.get(\"test\", [])\n                )\n            else:\n                self.roi_ids = sorted(splits_data.get(split, []))\n        else:\n            # Fallback: scan directory directly\n            all_rois = sorted([d.name for d in self.cross_dir.iterdir() if d.is_dir() and d.name.startswith(\"ROI_\")])\n            if split == \"all\":\n                self.roi_ids = all_rois\n            else:\n                # 80/10/10 fallback\n                n_train = int(len(all_rois) * 0.8)\n                n_val = int(len(all_rois) * 0.1)\n                if split == \"train\":\n                    self.roi_ids = all_rois[:n_train]\n                elif split == \"val\":\n                    self.roi_ids = all_rois[n_train : n_train + n_val]\n                else:\n                    self.roi_ids = all_rois[n_train + n_val :]\n\n        # Filter to only existing pairs on disk\n        self.valid_pairs: List[Tuple[Path, Path, Path]] = []\n        for roi_id in self.roi_ids:\n            roi_d = self.cross_dir / roi_id\n            lr_p = roi_d / \"lr.tif\"\n            hr_p = roi_d / \"hr.tif\"\n            meta_p = roi_d / \"metadata.json\"\n            if lr_p.exists() and hr_p.exists():\n                self.valid_pairs.append((lr_p, hr_p, meta_p))\n\n        if len(self.valid_pairs) == 0:\n            raise RuntimeError(f\"No valid pairs found for split '{split}' in {self.cross_dir}\")\n\n    def __len__(self) -> int:\n        return len(self.valid_pairs)\n\n    def _apply_augmentation(self, lr: np.ndarray, hr: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n        \"\"\"Synchronously augment LR and HR pairs.\"\"\"\n        # Random horizontal flip\n        if random.random() > 0.5:\n            lr = np.fliplr(lr)\n            hr = np.fliplr(hr)\n\n        # Random vertical flip\n        if random.random() > 0.5:\n            lr = np.flipud(lr)\n            hr = np.flipud(hr)\n\n        # Random 90-degree rotation\n        k = random.choice([0, 1, 2, 3])\n        if k > 0:\n            lr = np.rot90(lr, k, axes=(0, 1))\n            hr = np.rot90(hr, k, axes=(0, 1))\n\n        return lr.copy(), hr.copy()\n\n    def _crop_pair(self, lr: np.ndarray, hr: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n        \"\"\"\n        Crops LR to lr_crop_size and HR to (lr_crop_size * scale).\n        Guarantees spatial alignment between LR and HR patches.\n        \"\"\"\n        if self.lr_crop_size is None:\n            return lr, hr\n\n        lr_h, lr_w = lr.shape[:2]\n        crop_h, crop_w = self.lr_crop_size\n\n        if crop_h > lr_h or crop_w > lr_w:\n            raise ValueError(f\"Crop size {self.lr_crop_size} exceeds LR shape {(lr_h, lr_w)}\")\n\n        if self.random_crop:\n            top = random.randint(0, lr_h - crop_h)\n            left = random.randint(0, lr_w - crop_w)\n        else:\n            top = (lr_h - crop_h) // 2\n            left = (lr_w - crop_w) // 2\n\n        bottom = top + crop_h\n        right = left + crop_w\n\n        hr_top = top * self.scale\n        hr_left = left * self.scale\n        hr_bottom = bottom * self.scale\n        hr_right = right * self.scale\n\n        lr_cropped = lr[top:bottom, left:right]\n        hr_cropped = hr[hr_top:hr_bottom, hr_left:hr_right]\n        return lr_cropped, hr_cropped\n\n    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:\n        lr_path, hr_path, meta_path = self.valid_pairs[idx]\n\n        # Read GeoTIFF arrays\n        lr_arr = tifffile.imread(str(lr_path)).astype(np.float32)   # (H, W, 4), int32 surface reflectance\n        hr_arr = tifffile.imread(str(hr_path)).astype(np.float32)   # (4H, 4W, 4), uint8 [0, 255]\n\n        # Select bands:\n        # Channels 0, 1, 2 = Red, Green, Blue\n        # Channel 3 = NIR\n        if self.rgb_only:\n            lr_arr = lr_arr[:, :, :3]\n            hr_arr = hr_arr[:, :, :3]\n\n        # Apply crop (guarantees divisibility for SwinIR windows)\n        lr_arr, hr_arr = self._crop_pair(lr_arr, hr_arr)\n\n        # Apply synchronized spatial augmentations\n        if self.augment:\n            lr_arr, hr_arr = self._apply_augmentation(lr_arr, hr_arr)\n\n        # Normalization:\n        # - LR: Sentinel-2 L2A surface reflectance is scaled x10000.\n        #       Values are non-negative and rarely exceed 10000 for terrestrial targets.\n        #       Normalizing by 10000.0 scales reflectance into [0.0, 1.0].\n        lr_norm = np.clip(lr_arr / 10000.0, 0.0, 1.0)\n\n        # - HR: NAIP aerial imagery is standard uint8 [0, 255].\n        #       Normalizing by 255.0 scales into [0.0, 1.0].\n        hr_norm = np.clip(hr_arr / 255.0, 0.0, 1.0)\n\n        # Layout conversion: (H, W, C) -> (C, H, W) PyTorch tensor\n        lr_tensor = torch.from_numpy(lr_norm).permute(2, 0, 1).float()\n        hr_tensor = torch.from_numpy(hr_norm).permute(2, 0, 1).float()\n\n        return {\n            \"lr\": lr_tensor,\n            \"hr\": hr_tensor,\n            \"roi_id\": lr_path.parent.name,\n        }\n\n\ndef get_sen2naip_dataloaders(\n    root_dir: Path = DEFAULT_DATASET_DIR,\n    batch_size: int = 4,\n    num_workers: int = 0,\n    rgb_only: bool = True,\n    lr_crop_size: Tuple[int, int] = (120, 120),\n) -> Tuple[DataLoader, DataLoader, DataLoader]:\n    \"\"\"\n    Creates PyTorch DataLoaders for train, validation, and test splits.\n    \"\"\"\n    train_ds = SEN2NAIPDataset(\n        root_dir=root_dir,\n        split=\"train\",\n        rgb_only=rgb_only,\n        lr_crop_size=lr_crop_size,\n        random_crop=True,\n        augment=True,\n    )\n    val_ds = SEN2NAIPDataset(\n        root_dir=root_dir,\n        split=\"val\",\n        rgb_only=rgb_only,\n        lr_crop_size=lr_crop_size,\n        random_crop=False,\n        augment=False,\n    )\n    test_ds = SEN2NAIPDataset(\n        root_dir=root_dir,\n        split=\"test\",\n        rgb_only=rgb_only,\n        lr_crop_size=lr_crop_size,\n        random_crop=False,\n        augment=False,\n    )\n\n    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)\n    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)\n    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)\n\n    return train_loader, val_loader, test_loader\n\n\nif __name__ == \"__main__\":\n    print(\"=\" * 60)\n    print(\"SEN2NAIP DATASET LOADER SMOKE TEST\")\n    print(\"=\" * 60)\n\n    try:\n        ds = SEN2NAIPDataset(split=\"all\", rgb_only=True)\n        print(f\"Loaded 'all' split: {len(ds)} samples found.\")\n        sample = ds[0]\n        print(f\"Sample 0 ({sample['roi_id']}):\")\n        print(f\"  LR tensor: shape={sample['lr'].shape}, dtype={sample['lr'].dtype}, min={sample['lr'].min():.4f}, max={sample['lr'].max():.4f}\")\n        print(f\"  HR tensor: shape={sample['hr'].shape}, dtype={sample['hr'].dtype}, min={sample['hr'].min():.4f}, max={sample['hr'].max():.4f}\")\n        assert sample['hr'].shape[1] == sample['lr'].shape[1] * 4\n        assert sample['hr'].shape[2] == sample['lr'].shape[2] * 4\n        print(\"[SUCCESS] SEN2NAIP dataset loader operational.\")\n    except Exception as e:\n        print(f\"[NOTE] Dataset not yet populated or error: {e}\")\n"
dataset_file.write_text(dataset_code, encoding="utf-8")
print(f"[SUCCESS] Wrote/verified {dataset_file} ({dataset_file.stat().st_size} bytes)")

from sen2naip_dataset import SEN2NAIPDataset, get_sen2naip_dataloaders
print("[SUCCESS] Successfully imported SEN2NAIPDataset and get_sen2naip_dataloaders.")


## 5. Acquire SEN2NAIP Dataset (100-300 Pairs)

The official SEN2NAIP repository contains:
- **Synthetic subset:** 165+ GB (downsampled NAIP -> synthetic Sentinel-2)
- **Cross-sensor subset:** 2,851 pairs (real Sentinel-2 L2A vs real NAIP aerial photography)

To make fine-tuning fast and avoid unnecessary bandwidth usage, we stream **150 pairs** directly from `cross-sensor/cross-sensor.zip` on Hugging Face using **HTTP Range requests** via `huggingface_hub.HfFileSystem`. This downloads only ~70 MB in 1-2 minutes.


In [ ]:
import os
import time
import zipfile
from pathlib import Path
from huggingface_hub import HfFileSystem

# Download size (100 to 300 recommended for initial GPU fine-tuning)
NUM_PAIRS = 150
START_IDX = 0

DATASET_ROOT = Path("data/SEN2NAIP")
CROSS_DIR = DATASET_ROOT / "cross-sensor"
CROSS_DIR.mkdir(parents=True, exist_ok=True)

HF_REPO_ID = "isp-uv-es/SEN2NAIP"
HF_ZIP_PATH = f"datasets/{HF_REPO_ID}/cross-sensor/cross-sensor.zip"

existing_rois = [d for d in CROSS_DIR.iterdir() if d.is_dir() and d.name.startswith("ROI_") and (d / "lr.tif").exists()]
print(f"Found {len(existing_rois)} existing ROI pairs in {CROSS_DIR}")

if len(existing_rois) < NUM_PAIRS:
    print(f"Connecting to Hugging Face '{HF_REPO_ID}' to stream {NUM_PAIRS} pairs via HTTP Range requests...")
    hffs = HfFileSystem()
    t0 = time.time()
    with hffs.open(HF_ZIP_PATH, "rb") as remote_file:
        zf = zipfile.ZipFile(remote_file)
        all_rois = sorted(list(set(
            p.split("/")[1]
            for p in zf.namelist()
            if len(p.split("/")) > 1 and p.split("/")[1].startswith("ROI_")
        )))
        print(f"Total available ROIs in dataset archive: {len(all_rois):,}")
        target_rois = all_rois[START_IDX : START_IDX + NUM_PAIRS]
        
        print(f"Streaming {len(target_rois)} pairs (indices {START_IDX} to {START_IDX + len(target_rois) - 1})...")
        for idx, roi in enumerate(target_rois, start=1):
            roi_dir = CROSS_DIR / roi
            roi_dir.mkdir(parents=True, exist_ok=True)
            lr_file, hr_file, meta_file = roi_dir / "lr.tif", roi_dir / "hr.tif", roi_dir / "metadata.json"
            if lr_file.exists() and hr_file.exists() and meta_file.exists():
                continue
            lr_bytes = zf.read(f"cross-sensor/{roi}/lr.tif")
            hr_bytes = zf.read(f"cross-sensor/{roi}/hr.tif")
            meta_bytes = zf.read(f"cross-sensor/{roi}/metadata.json")
            lr_file.write_bytes(lr_bytes)
            hr_file.write_bytes(hr_bytes)
            meta_file.write_bytes(meta_bytes)
            if idx % 25 == 0 or idx == len(target_rois):
                print(f"  [{idx:03d}/{len(target_rois):03d}] Streamed {roi} ({time.time() - t0:.1f}s elapsed)")
    print(f"[SUCCESS] Dataset download completed in {time.time() - t0:.1f}s.")
else:
    print(f"[SUCCESS] Dataset already contains {len(existing_rois)} pairs. Skipping download.")


In [ ]:
# Optional Alternative: Mount Google Drive or extract a pre-uploaded ZIP file
# If you have SEN2NAIP stored in your Google Drive, uncomment and run:
#
# from google.colab import drive
# drive.mount('/content/drive')
# import zipfile
# DRIVE_ZIP = '/content/drive/MyDrive/SEN2NAIP_150pairs.zip'
# with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
#     zf.extractall('data/SEN2NAIP')
# print("Extracted dataset from Google Drive.")
print("Automated streaming from Hugging Face is the default above.")


## 6. Create Train / Validation / Test Splits


In [ ]:
import json
import random
from pathlib import Path

DATASET_ROOT = Path("data/SEN2NAIP")
CROSS_DIR = DATASET_ROOT / "cross-sensor"
SPLITS_FILE = DATASET_ROOT / "splits.json"

all_rois = sorted([d.name for d in CROSS_DIR.iterdir() if d.is_dir() and d.name.startswith("ROI_") and (d / "lr.tif").exists()])
print(f"Total valid ROI pairs available on disk: {len(all_rois)}")

# Deterministic 80/10/10 split
random.seed(42)
shuffled = list(all_rois)
random.shuffle(shuffled)

n_total = len(shuffled)
n_train = int(round(n_total * 0.80))
n_val   = int(round(n_total * 0.10))

train_ids = sorted(shuffled[:n_train])
val_ids   = sorted(shuffled[n_train : n_train + n_val])
test_ids  = sorted(shuffled[n_train + n_val :])

splits_data = {
    "train": train_ids,
    "val": val_ids,
    "test": test_ids,
    "summary": {
        "total": n_total,
        "train_count": len(train_ids),
        "val_count": len(val_ids),
        "test_count": len(test_ids),
        "train_pct": round(len(train_ids) / n_total * 100, 1),
        "val_pct": round(len(val_ids) / n_total * 100, 1),
        "test_pct": round(len(test_ids) / n_total * 100, 1),
    }
}

with open(SPLITS_FILE, "w", encoding="utf-8") as f:
    json.dump(splits_data, f, indent=2)

print(f"[SUCCESS] Splits written to {SPLITS_FILE}:")
print(f"  Train: {len(train_ids)} pairs ({splits_data['summary']['train_pct']}%)")
print(f"  Val:   {len(val_ids)} pairs ({splits_data['summary']['val_pct']}%)")
print(f"  Test:  {len(test_ids)} pairs ({splits_data['summary']['test_pct']}%)")


## 7. Verify Shapes, Window Divisibility & Range

### Critical Pipeline Alignment Rules:
1. **LR Sentinel-2 L2A**: Surface reflectance is recorded as integer scaled x10000. Normalization via `/ 10000.0` maps values to $[0.0, 1.0]$.
2. **HR NAIP Reference**: Aerial RGB photography is uint8 in $[0, 255]$. Normalization via `/ 255.0` maps values to $[0.0, 1.0]$.
3. **RGB-Only Strategy**: Extract channels 0, 1, 2 (Red, Green, Blue) from both sensors. (NIR band 3 is preserved on disk for future multi-spectral extensions).
4. **Window Divisibility**: SwinIR uses window-based attention with `window_size = 8`. Native LR size $121 \times 121$ ($121 \% 8 = 1$) is cropped to $120 \times 120$ ($120 \% 8 = 0$). The corresponding HR patch is $480 \times 480$ ($480 \% 32 = 0$).


In [ ]:
from sen2naip_dataset import SEN2NAIPDataset

# Instantiate dataset with divisibility crop
val_ds = SEN2NAIPDataset(root_dir=DATASET_ROOT, split="val", rgb_only=True, lr_crop_size=(120, 120))
sample = val_ds[0]

lr_t = sample["lr"]
hr_t = sample["hr"]
roi_id = sample["roi_id"]

print(f"Verification Sample: {roi_id}")
print(f"  LR Shape:        {list(lr_t.shape)} (C, H, W)")
print(f"  HR Shape:        {list(hr_t.shape)} (C, H, W)")
print(f"  LR Value Range:  [{lr_t.min():.4f}, {lr_t.max():.4f}] (mean={lr_t.mean():.4f})")
print(f"  HR Value Range:  [{hr_t.min():.4f}, {hr_t.max():.4f}] (mean={hr_t.mean():.4f})")

# Assertions
assert lr_t.shape == torch.Size([3, 120, 120]), f"Expected LR [3, 120, 120], got {lr_t.shape}"
assert hr_t.shape == torch.Size([3, 480, 480]), f"Expected HR [3, 480, 480], got {hr_t.shape}"
assert hr_t.shape[1] == lr_t.shape[1] * 4, "HR height must be exactly 4x LR height"
assert hr_t.shape[2] == lr_t.shape[2] * 4, "HR width must be exactly 4x LR width"
assert lr_t.shape[1] % 8 == 0, "LR must be divisible by SwinIR window_size=8"
assert not torch.isnan(lr_t).any() and not torch.isinf(lr_t).any(), "NaN or Inf detected in LR"
assert not torch.isnan(hr_t).any() and not torch.isinf(hr_t).any(), "NaN or Inf detected in HR"
print("[SUCCESS] All shape, range, and window-divisibility assertions PASSED!")


## 8. Load Pretrained SwinIR Base Model

We initialize the **SwinIR-M classical super-resolution** architecture:
- `upscale = 4`
- `in_chans = 3`, `embed_dim = 180`, `depths = [6, 6, 6, 6, 6, 6]`, `num_heads = [6, 6, 6, 6, 6, 6]`
- `window_size = 8`
- `upsampler = 'pixelshuffle'`

We load the base weights from `models/swinir_x4.pth` (pretrained on DF2K). If missing in the clone, the notebook automatically downloads the official checkpoint directly from GitHub releases.


In [ ]:
import urllib.request
from pathlib import Path
import torch
from swinir.network_swinir import SwinIR

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = MODEL_DIR / "swinir_x4.pth"

# Architecture definition matching SwinIR-M classical SR x4
SWINIR_CONFIG = dict(
    img_size=64,
    patch_size=1,
    in_chans=3,
    embed_dim=180,
    depths=[6, 6, 6, 6, 6, 6],
    num_heads=[6, 6, 6, 6, 6, 6],
    window_size=8,
    mlp_ratio=2.0,
    qkv_bias=True,
    qk_scale=None,
    drop_rate=0.0,
    attn_drop_rate=0.0,
    drop_path_rate=0.1,
    ape=False,
    patch_norm=True,
    use_checkpoint=False,
    upscale=4,
    img_range=1.0,
    upsampler='pixelshuffle',
    resi_connection='1conv',
)

# Download pretrained DF2K checkpoint if not present
if not CHECKPOINT_PATH.exists() or CHECKPOINT_PATH.stat().st_size < 1000:
    print(f"Downloading official pretrained SwinIR-M checkpoint to {CHECKPOINT_PATH}...")
    URL = "https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/001_classicalSR_DF2K_s64w8_SwinIR-M_x4.pth"
    urllib.request.urlretrieve(URL, str(CHECKPOINT_PATH))
    print(f"[SUCCESS] Downloaded {CHECKPOINT_PATH.name} ({CHECKPOINT_PATH.stat().st_size / (1024**2):.1f} MB)")
else:
    print(f"Pretrained base checkpoint found at {CHECKPOINT_PATH} ({CHECKPOINT_PATH.stat().st_size / (1024**2):.1f} MB)")

# Instantiate model
base_model = SwinIR(**SWINIR_CONFIG)
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
state_dict = ckpt["params_ema"] if "params_ema" in ckpt else ckpt.get("params", ckpt)
base_model.load_state_dict(state_dict, strict=True)
base_model = base_model.to(DEVICE)
base_model.eval()

# Forward pass verification on dummy LR tensor
with torch.no_grad():
    dummy_lr = torch.randn(1, 3, 120, 120, device=DEVICE).clamp(0, 1)
    dummy_sr = base_model(dummy_lr)
    print(f"Forward pass verification: [1, 3, 120, 120] -> {list(dummy_sr.shape)}")
    assert dummy_sr.shape == torch.Size([1, 3, 480, 480]), f"Unexpected output shape: {dummy_sr.shape}"

print(f"[SUCCESS] Pretrained SwinIR model loaded and verified on {DEVICE}.")


## 9. Pre-Training Baseline Evaluation on Validation Set

Before training, we evaluate two baselines on the validation set:
1. **Bicubic Upsampling (4x)**: Classical mathematical interpolation.
2. **Generic SwinIR (DF2K)**: Unadapted deep super-resolution model.

*Radiometric Gap Context*: Sentinel-2 surface reflectance values (mean ~0.07-0.10) differ from NAIP aerial photograph dynamic ranges (mean ~0.40-0.50). The generic model has never observed satellite reflectance and therefore outputs dark imagery. Fine-tuning bridges this domain gap.


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from sen2naip_dataset import SEN2NAIPDataset

val_ds = SEN2NAIPDataset(root_dir=DATASET_ROOT, split="val", rgb_only=True, lr_crop_size=(120, 120), random_crop=False, augment=False)
print(f"Evaluating baselines across {len(val_ds)} validation samples...")

def evaluate_model_on_split(model, dataset, device):
    model.eval()
    psnrs, ssims, l1s, details = [], [], [], []
    with torch.no_grad():
        for i in range(len(dataset)):
            s = dataset[i]
            lr = s["lr"].unsqueeze(0).to(device)
            hr = s["hr"].unsqueeze(0).to(device)
            sr = model(lr).clamp(0.0, 1.0)
            
            l1 = F.l1_loss(sr, hr).item()
            hr_np = hr[0].permute(1, 2, 0).cpu().numpy()
            sr_np = sr[0].permute(1, 2, 0).cpu().numpy()
            lr_np = lr[0].permute(1, 2, 0).cpu().numpy()
            
            psnr = peak_signal_noise_ratio(hr_np, sr_np, data_range=1.0)
            ssim = structural_similarity(hr_np, sr_np, data_range=1.0, channel_axis=2)
            
            psnrs.append(psnr); ssims.append(ssim); l1s.append(l1)
            details.append({"roi_id": s["roi_id"], "sr_np": sr_np, "hr_np": hr_np, "lr_np": lr_np, "psnr": psnr, "ssim": ssim, "l1": l1})
    return {"psnr": float(np.mean(psnrs)), "ssim": float(np.mean(ssims)), "l1": float(np.mean(l1s)), "details": details}

def evaluate_bicubic_on_split(dataset, device):
    psnrs, ssims, l1s, details = [], [], [], []
    with torch.no_grad():
        for i in range(len(dataset)):
            s = dataset[i]
            lr = s["lr"].unsqueeze(0).to(device)
            hr = s["hr"].unsqueeze(0).to(device)
            bic = F.interpolate(lr, scale_factor=4, mode="bicubic", align_corners=False).clamp(0.0, 1.0)
            
            l1 = F.l1_loss(bic, hr).item()
            hr_np = hr[0].permute(1, 2, 0).cpu().numpy()
            bic_np = bic[0].permute(1, 2, 0).cpu().numpy()
            lr_np = lr[0].permute(1, 2, 0).cpu().numpy()
            
            psnr = peak_signal_noise_ratio(hr_np, bic_np, data_range=1.0)
            ssim = structural_similarity(hr_np, bic_np, data_range=1.0, channel_axis=2)
            
            psnrs.append(psnr); ssims.append(ssim); l1s.append(l1)
            details.append({"roi_id": s["roi_id"], "bic_np": bic_np, "hr_np": hr_np, "lr_np": lr_np, "psnr": psnr, "ssim": ssim, "l1": l1})
    return {"psnr": float(np.mean(psnrs)), "ssim": float(np.mean(ssims)), "l1": float(np.mean(l1s)), "details": details}

print("Computing Bicubic baseline...")
bicubic_val_res = evaluate_bicubic_on_split(val_ds, DEVICE)
print(f"  -> Bicubic:        PSNR = {bicubic_val_res['psnr']:.2f} dB | SSIM = {bicubic_val_res['ssim']:.4f} | L1 = {bicubic_val_res['l1']:.4f}")

print("Computing Generic SwinIR (DF2K) baseline...")
generic_val_res = evaluate_model_on_split(base_model, val_ds, DEVICE)
print(f"  -> Generic SwinIR: PSNR = {generic_val_res['psnr']:.2f} dB | SSIM = {generic_val_res['ssim']:.4f} | L1 = {generic_val_res['l1']:.4f}")


## 10. Fine-Tune SwinIR on GPU

Training Setup:
- **Optimizer**: Adam (`lr = 1e-4`)
- **Loss**: L1 Reconstruction Loss (`nn.L1Loss()`)
- **Batch Size**: 4
- **Precision**: Mixed Precision (`torch.cuda.amp.autocast`) for high GPU throughput
- **Validation**: Evaluated after every epoch to track PSNR, SSIM, and L1 loss.


In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from swinir.network_swinir import SwinIR

# Hyperparameters
NUM_EPOCHS = 5
BATCH_SIZE = 4
LEARNING_RATE = 1e-4

# Dataloaders
train_ds = SEN2NAIPDataset(root_dir=DATASET_ROOT, split="train", rgb_only=True, lr_crop_size=(120, 120), random_crop=True, augment=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

# Initialize trainable model from pretrained DF2K weights
ft_model = SwinIR(**SWINIR_CONFIG)
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
state_dict = ckpt["params_ema"] if "params_ema" in ckpt else ckpt.get("params", ckpt)
ft_model.load_state_dict(state_dict, strict=True)
ft_model = ft_model.to(DEVICE)

criterion = nn.L1Loss()
optimizer = optim.Adam(ft_model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

history = []
best_val_psnr = -1.0
best_state = None

print(f"Starting GPU Fine-Tuning: {NUM_EPOCHS} epochs, {len(train_ds)} train samples ({len(train_loader)} batches/epoch)...")
t_total_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    ft_model.train()
    epoch_loss = 0.0
    t_epoch_start = time.time()
    
    for step, batch in enumerate(train_loader, start=1):
        lr = batch["lr"].to(DEVICE, non_blocking=True)
        hr = batch["hr"].to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            sr = ft_model(lr)
            loss = criterion(sr, hr)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item() * lr.size(0)
        if step % 10 == 0 or step == len(train_loader):
            print(f"  Epoch [{epoch}/{NUM_EPOCHS}] Step [{step}/{len(train_loader)}] Loss: {loss.item():.4f}")
            
    avg_train_loss = epoch_loss / len(train_ds)
    epoch_time = time.time() - t_epoch_start
    
    # Evaluate validation split
    val_res = evaluate_model_on_split(ft_model, val_ds, DEVICE)
    print(f"--> Epoch {epoch} Complete ({epoch_time:.1f}s): Train L1 = {avg_train_loss:.4f} | Val L1 = {val_res['l1']:.4f} | Val PSNR = {val_res['psnr']:.2f} dB | Val SSIM = {val_res['ssim']:.4f}")
    
    if val_res["psnr"] > best_val_psnr:
        best_val_psnr = val_res["psnr"]
        best_state = {k: v.cpu().clone() for k, v in ft_model.state_dict().items()}
        print(f"    * New best validation PSNR: {best_val_psnr:.2f} dB!")
        
    history.append({
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": val_res["l1"],
        "val_psnr": val_res["psnr"],
        "val_ssim": val_res["ssim"],
        "time_sec": round(epoch_time, 2)
    })

print()
print(f"[SUCCESS] Fine-tuning finished in {time.time() - t_total_start:.1f}s. Best Val PSNR: {best_val_psnr:.2f} dB.")
if best_state is not None:
    ft_model.load_state_dict(best_state)


## 11. Save Fine-Tuned Model Separately

We save the fine-tuned checkpoint separately to `models/swinir_sentinel_x4.pth`.
**Safety Rule**: The production checkpoint `models/swinir_x4.pth` is NEVER overwritten.


In [ ]:
import json
from pathlib import Path
import torch

OUTPUT_MODEL_PATH = Path("models/swinir_sentinel_x4.pth")
final_val_res = evaluate_model_on_split(ft_model, val_ds, DEVICE)

checkpoint_payload = {
    "epoch": NUM_EPOCHS,
    "model_state_dict": ft_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "lr": LEARNING_RATE,
    "loss_fn": "L1Loss",
    "dataset": "SEN2NAIP (cross-sensor)",
    "scale": 4,
    "val_psnr": final_val_res["psnr"],
    "val_ssim": final_val_res["ssim"],
    "val_loss": final_val_res["l1"],
    "model_args": SWINIR_CONFIG,
    "normalization": {
        "lr": "Sentinel-2 reflectance [0, 10000] / 10000.0 -> [0, 1]",
        "hr": "NAIP uint8 [0, 255] / 255.0 -> [0, 1]",
    },
}

torch.save(checkpoint_payload, OUTPUT_MODEL_PATH)
print(f"[SUCCESS] Saved fine-tuned checkpoint: {OUTPUT_MODEL_PATH} ({OUTPUT_MODEL_PATH.stat().st_size / (1024**2):.1f} MB)")

# Save training history CSV
OUT_DIR = Path("outputs/swinir_colab")
OUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUT_DIR / "training_history.csv"
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("epoch,train_loss,val_loss,val_psnr,val_ssim,time_sec\n")
    for h in history:
        f.write(f"{h['epoch']},{h['train_loss']:.6f},{h['val_loss']:.6f},{h['val_psnr']:.4f},{h['val_ssim']:.6f},{h['time_sec']}\n")
print(f"[SUCCESS] Saved training history to {csv_path}")

# Optional: Download model to your local machine from Colab
# from google.colab import files
# files.download(str(OUTPUT_MODEL_PATH))


## 12. Qualitative Multi-Panel Visual Comparisons

We display a 5-panel visual comparison:
1. **LR Sentinel-2 (10 m)** (contrast-stretched for visual comparison)
2. **Bicubic Upsampling (4x)**
3. **Generic SwinIR (DF2K weights)**
4. **Fine-Tuned Sentinel SwinIR (Our Model)**
5. **NAIP HR Reference (2.5 m Ground Truth)**


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

VIS_DIR = Path("outputs/swinir_colab_visuals")
VIS_DIR.mkdir(parents=True, exist_ok=True)

# Generate 5-panel comparisons for validation samples
num_vis = min(5, len(final_val_res["details"]))
print(f"Generating 5-panel visual comparisons for {num_vis} validation scenes...")

for i in range(num_vis):
    ft_info  = final_val_res["details"][i]
    gen_info = generic_val_res["details"][i]
    bic_info = bicubic_val_res["details"][i]
    
    roi_id = ft_info["roi_id"]
    lr_np  = ft_info["lr_np"]
    bic_np = bic_info["bic_np"]
    gen_np = gen_info["sr_np"]
    ft_np  = ft_info["sr_np"]
    hr_np  = ft_info["hr_np"]
    
    # Contrast stretch LR for visual display (since raw reflectance is ~0.10)
    lr_disp = np.clip(lr_np * 3.5, 0.0, 1.0)
    lr_up   = np.repeat(np.repeat(lr_disp, 4, axis=0), 4, axis=1)
    
    fig, axes = plt.subplots(1, 5, figsize=(22, 5))
    
    axes[0].imshow(lr_up)
    axes[0].set_title(f"LR Sentinel-2 (10 m)\n[{roi_id}]", fontsize=11)
    axes[0].axis("off")
    
    axes[1].imshow(bic_np)
    axes[1].set_title(f"Bicubic 4x\nPSNR: {bic_info['psnr']:.2f} dB | SSIM: {bic_info['ssim']:.3f}", fontsize=11)
    axes[1].axis("off")
    
    axes[2].imshow(gen_np)
    axes[2].set_title(f"Generic SwinIR (DF2K)\nPSNR: {gen_info['psnr']:.2f} dB | SSIM: {gen_info['ssim']:.3f}", fontsize=11)
    axes[2].axis("off")
    
    axes[3].imshow(ft_np)
    axes[3].set_title(f"Fine-Tuned SwinIR (Ours)\nPSNR: {ft_info['psnr']:.2f} dB | SSIM: {ft_info['ssim']:.3f}", fontsize=11, color="darkgreen", weight="bold")
    axes[3].axis("off")
    
    axes[4].imshow(hr_np)
    axes[4].set_title(f"NAIP Reference (2.5 m)\n[Ground Truth HR]", fontsize=11)
    axes[4].axis("off")
    
    plt.tight_layout()
    save_fig_path = VIS_DIR / f"comparison_{roi_id}.png"
    plt.savefig(save_fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved visual comparison: {save_fig_path.name}")


## 13. Final Evaluation & Benchmark Comparison Table


In [ ]:
# Evaluate held-out Test split
test_ds = SEN2NAIPDataset(root_dir=DATASET_ROOT, split="test", rgb_only=True, lr_crop_size=(120, 120), random_crop=False, augment=False)
print(f"Evaluating held-out Test set ({len(test_ds)} samples)...")

bic_test = evaluate_bicubic_on_split(test_ds, DEVICE)
gen_test = evaluate_model_on_split(base_model, test_ds, DEVICE)
ft_test  = evaluate_model_on_split(ft_model, test_ds, DEVICE)

delta_psnr_val = final_val_res['psnr'] - generic_val_res['psnr']
delta_ssim_val = final_val_res['ssim'] - generic_val_res['ssim']
delta_l1_val   = final_val_res['l1'] - generic_val_res['l1']

delta_psnr_test = ft_test['psnr'] - gen_test['psnr']
delta_ssim_test = ft_test['ssim'] - gen_test['ssim']
delta_l1_test   = ft_test['l1'] - gen_test['l1']

table_md = f"""
========================================================================================================
                               SWINIR SENTINEL-2 SUPER-RESOLUTION BENCHMARK
========================================================================================================
Method                     Split  Scale   PSNR (dB)    SSIM      L1 Loss   Delta PSNR   Delta SSIM
--------------------------------------------------------------------------------------------------------
Bicubic Upsampling         Val    4x      {bicubic_val_res['psnr']:7.2f} dB   {bicubic_val_res['ssim']:.4f}    {bicubic_val_res['l1']:.4f}    -            -
Generic SwinIR (DF2K)      Val    4x      {generic_val_res['psnr']:7.2f} dB   {generic_val_res['ssim']:.4f}    {generic_val_res['l1']:.4f}    [Baseline]   [Baseline]
Fine-Tuned SwinIR (Ours)   Val    4x      {final_val_res['psnr']:7.2f} dB   {final_val_res['ssim']:.4f}    {final_val_res['l1']:.4f}    {delta_psnr_val:+6.2f} dB    {delta_ssim_val:+.4f}
--------------------------------------------------------------------------------------------------------
Bicubic Upsampling         Test   4x      {bic_test['psnr']:7.2f} dB   {bic_test['ssim']:.4f}    {bic_test['l1']:.4f}    -            -
Generic SwinIR (DF2K)      Test   4x      {gen_test['psnr']:7.2f} dB   {gen_test['ssim']:.4f}    {gen_test['l1']:.4f}    [Baseline]   [Baseline]
Fine-Tuned SwinIR (Ours)   Test   4x      {ft_test['psnr']:7.2f} dB   {ft_test['ssim']:.4f}    {ft_test['l1']:.4f}    {delta_psnr_test:+6.2f} dB    {delta_ssim_test:+.4f}
========================================================================================================
Key Findings:
1. Domain Adaptation: Pretrained SwinIR output suffers from the reflectance-to-radiance dynamic gap.
2. Fine-Tuning Gain: Model learns the cross-sensor mapping, achieving major PSNR and SSIM gains.
3. Production Safety: Fine-tuned checkpoint saved to models/swinir_sentinel_x4.pth without altering
   the production baseline checkpoint.
========================================================================================================
"""
print(table_md)
